# 23CSE301 — Machine Learning Capstone | Review 1

## Predicting Metro Interstate Traffic Volume using Regression

---

### Problem Statement

Given hourly meteorological, temporal, and holiday attributes — **temperature, rain, snow, cloud coverage, weather conditions, date/time, and national holidays** — build and evaluate regression models to accurately predict **Interstate 94 Westbound hourly traffic volume** (`traffic_volume`).

This is a **supervised regression** task:
* **Input X:** 9 features (weather conditions, temperature, and temporal attributes extracted from datetime)
* **Output y:** `traffic_volume` (vehicles per hour) — a continuous numerical target variable

### Dataset at a Glance

| Field | Value |
|-------|-------|
| File | `../dataset/Metro_Interstate_Traffic_Volume.csv` |
| Raw shape | 48 204 rows × 9 columns |
| Source | UCI Machine Learning Repository / Kaggle |

| Column | dtype | Role | Description |
|--------|-------|------|-------------|
| `holiday` | object | Feature | Categorical indicator of US national holidays |
| `temp` | float64 | Feature | Temperature in Kelvin |
| `rain_1h` | float64 | Feature | Amount in mm of rain that occurred in the hour |
| `snow_1h` | float64 | Feature | Amount in mm of snow that occurred in the hour |
| `clouds_all` | int64 | Feature | Percentage of cloud cover (0–100%) |
| `weather_main` | object | Feature | Short textual description of weather |
| `weather_description` | object | Feature | Detailed textual description of weather |
| `date_time` | object | Feature | Hourly timestamp (YYYY-MM-DD HH:MM:SS) |
| `traffic_volume` | int64 | **Regression target** | Hourly traffic volume (vehicles/hour) |

### Project Objectives
1. Audit, clean, and explore the dataset.
2. Engineer temporal and environmental features (`hour`, `day_of_week`, `month`, `is_weekend`, `is_holiday`, `temp_celsius`).
3. Train and compare **the first 5 regression algorithms** on the same held-out test set:
   1. **Linear Regression**
   2. **Ridge Regression**
   3. **Lasso Regression**
   4. **ElasticNet Regression**
   5. **Polynomial Regression (degree = 2)**
4. Tune hyperparameters for the regularized linear models using 5-fold cross-validation.
5. Evaluate model performance using R², RMSE, and MAE, and analyze residual distributions.

## Section 2 — Imports and Configuration

In [1]:
# Standard library
import warnings
warnings.filterwarnings('ignore')

# Data processing
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# IPython display
from IPython.display import display

# Preprocessing & pipelines
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer

# Model selection
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score

# Regression models (First 5 Algorithms)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

# Metrics
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Global configuration
RANDOM_STATE = 42
TEST_SIZE = 0.20

sns.set_theme(style='whitegrid', palette='husl')
pd.set_option('display.float_format', '{:.4f}'.format)

print(f"All imports successful.")
print(f"RANDOM_STATE = {RANDOM_STATE}  |  TEST_SIZE = {TEST_SIZE}")

All imports successful.
RANDOM_STATE = 42  |  TEST_SIZE = 0.2


## Section 3 — Dataset Loading and Audit

Raw data is loaded from the dataset directory without modification.
This section is a **read-only inspection** to check shape, data types, missing values, duplicates, and summary statistics.

In [2]:
df_raw = pd.read_csv('../dataset/Metro_Interstate_Traffic_Volume.csv')
print(f"Shape: {df_raw.shape[0]} rows  x  {df_raw.shape[1]} columns")

Shape: 48204 rows  x  9 columns


In [3]:
df_raw.head(10)

In [4]:
# Data types and non-null counts
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   holiday              61 non-null     str    
 1   temp                 48204 non-null  float64
 2   rain_1h              48204 non-null  float64
 3   snow_1h              48204 non-null  float64
 4   clouds_all           48204 non-null  int64  
 5   weather_main         48204 non-null  str    
 6   weather_description  48204 non-null  str    
 7   date_time            48204 non-null  str    
 8   traffic_volume       48204 non-null  int64  
dtypes: float64(3), int64(2), str(4)
memory usage: 3.3 MB


In [5]:
# Missing-value audit
missing  = df_raw.isnull().sum().rename('Missing Count')
miss_pct = (df_raw.isnull().mean() * 100).rename('Missing %')
display(pd.concat([df_raw.dtypes.rename('dtype'), missing, miss_pct], axis=1))

                       dtype  Missing Count  Missing %
holiday                  str          48143    99.8735
temp                 float64              0     0.0000
rain_1h              float64              0     0.0000
snow_1h              float64              0     0.0000
clouds_all             int64              0     0.0000
weather_main             str              0     0.0000
weather_description      str              0     0.0000
date_time                str              0     0.0000
traffic_volume         int64              0     0.0000


In [6]:
# Duplicate-row audit
n_dup = df_raw.duplicated().sum()
print(f"Exact duplicate rows: {n_dup}")
if n_dup > 0:
    display(df_raw[df_raw.duplicated(keep=False)].head(10))

Exact duplicate rows: 17
      holiday     temp  ...            date_time  traffic_volume
18696     NaN 286.2900  ...  2015-09-30 19:00:00            3679
18697     NaN 286.2900  ...  2015-09-30 19:00:00            3679
23850     NaN 289.0600  ...  2016-06-01 10:00:00            4831
23851     NaN 289.0600  ...  2016-06-01 10:00:00            4831
26783     NaN 289.7750  ...  2016-09-21 15:00:00            5365
26784     NaN 289.7750  ...  2016-09-21 15:00:00            5365
26979     NaN 287.8600  ...  2016-09-29 19:00:00            3435
26980     NaN 287.8600  ...  2016-09-29 19:00:00            3435
27170     NaN 279.2870  ...  2016-10-07 18:00:00            4642
27171     NaN 279.2870  ...  2016-10-07 18:00:00            4642

[10 rows x 9 columns]


In [7]:
# Descriptive statistics — numerical columns
df_raw.describe().T

In [8]:
# Unique values in categorical columns
for col in ['holiday', 'weather_main']:
    print(f"\n{col}  ({df_raw[col].nunique()} unique values)")
    print(df_raw[col].value_counts(dropna=False).head(10).to_string())


holiday  (11 unique values)
holiday
NaN                          48143
Labor Day                        7
Thanksgiving Day                 6
Christmas Day                    6
New Years Day                    6
Martin Luther King Jr Day        6
Columbus Day                     5
Veterans Day                     5
Washingtons Birthday             5
Memorial Day                     5

weather_main  (11 unique values)
weather_main
Clouds          15164
Clear           13391
Mist             5950
Rain             5672
Snow             2876
Drizzle          1821
Haze             1360
Thunderstorm     1034
Fog               912
Smoke              20


In [9]:
# Target-column summary
traffic = df_raw['traffic_volume']
target_stats = pd.Series({
    'Count'        : int(traffic.count()),
    'Min'          : traffic.min(),
    'Max'          : traffic.max(),
    'Mean'         : traffic.mean(),
    'Median'       : traffic.median(),
    'Std Dev'      : traffic.std(),
    'Skewness'     : traffic.skew(),
    'Kurtosis'     : traffic.kurtosis(),
    'Q1 (25%)'     : traffic.quantile(0.25),
    'Q3 (75%)'     : traffic.quantile(0.75),
    'IQR'          : traffic.quantile(0.75) - traffic.quantile(0.25),
}, name='traffic_volume')
display(target_stats.to_frame())

          traffic_volume
Count         48204.0000
Min               0.0000
Max            7280.0000
Mean           3259.8184
Median         3380.0000
Std Dev        1986.8607
Skewness         -0.0894
Kurtosis         -1.3091
Q1 (25%)       1193.0000
Q3 (75%)       4933.0000
IQR            3740.0000


## Section 4 — Exploratory Data Analysis (EDA)

### 4A — Distribution of Environmental & Numerical Features

In [10]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Environmental Feature Distributions', fontsize=15, fontweight='bold', y=1.01)

# temp
sns.histplot(df_raw[df_raw['temp'] > 100]['temp'] - 273.15, bins=30, kde=True, ax=axes[0, 0], color='crimson')
axes[0, 0].set(title='Temperature (°C)', xlabel='Temperature (°C)', ylabel='Count')

# rain_1h
sns.histplot(df_raw[df_raw['rain_1h'] < 50]['rain_1h'], bins=30, kde=True, ax=axes[0, 1], color='dodgerblue')
axes[0, 1].set(title='Rain in 1 hour (mm)', xlabel='Rain (mm)', ylabel='Count')

# clouds_all
sns.histplot(df_raw['clouds_all'], bins=20, kde=True, ax=axes[1, 0], color='teal')
axes[1, 0].set(title='Cloud Coverage (%)', xlabel='Clouds (%)', ylabel='Count')

# weather_main
sns.countplot(y='weather_main', data=df_raw, ax=axes[1, 1], palette='tab10', order=df_raw['weather_main'].value_counts().index)
axes[1, 1].set(title='Weather Main Category', xlabel='Count', ylabel='')

plt.tight_layout()
plt.show()

### 4B — Target Variable Distribution (`traffic_volume`)

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Target Variable — Hourly Traffic Volume Distribution', fontsize=14, fontweight='bold')

# Histogram + KDE
sns.histplot(df_raw['traffic_volume'], bins=50, kde=True, ax=axes[0], color='mediumpurple')
axes[0].axvline(df_raw['traffic_volume'].mean(), color='red', linestyle='--', label=f"Mean   {df_raw['traffic_volume'].mean():,.0f}")
axes[0].axvline(df_raw['traffic_volume'].median(), color='green', linestyle='--', label=f"Median {df_raw['traffic_volume'].median():,.0f}")
axes[0].set(title='Histogram + KDE', xlabel='Traffic Volume (vehicles/hour)', ylabel='Count')
axes[0].legend(fontsize=9)

# Box plot
sns.boxplot(y=df_raw['traffic_volume'], ax=axes[1], color='mediumpurple')
axes[1].set(title='Box Plot', ylabel='Traffic Volume (vehicles/hour)')

plt.tight_layout()
plt.show()

print(f"Skewness : {df_raw['traffic_volume'].skew():.4f}")
print(f"Mean     : {df_raw['traffic_volume'].mean():>12,.2f}")
print(f"Median   : {df_raw['traffic_volume'].median():>12,.2f}")

Skewness : -0.0894
Mean     :     3,259.82
Median   :     3,380.00


### 4C — Weather Category vs Traffic Volume

In [12]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(x='weather_main', y='traffic_volume', data=df_raw, ax=ax, palette='Set2')
ax.set_title('Traffic Volume by Main Weather Condition', fontsize=13, fontweight='bold')
ax.set_xlabel('Weather Condition', fontsize=11)
ax.set_ylabel('Traffic Volume (vehicles/hour)', fontsize=11)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Section 5 — Data Cleaning

Three steps:
1. Missing value treatment: `holiday` missing values indicate non-holiday days and are imputed with `'None'`.
2. Duplicate row removal: Exact duplicates (~17 rows) are dropped.
3. Outlier analysis & sensor anomaly removal: Temperature values of 0 K (absolute zero, physical sensor fault) and rain values > 1000 mm (recording artifact) are filtered out.

In [13]:
# Duplicate removal & missing handling
df_clean = df_raw.copy()
n_before = len(df_clean)

df_clean.drop_duplicates(keep='first', inplace=True)
df_clean.reset_index(drop=True, inplace=True)
df_clean['holiday'] = df_clean['holiday'].fillna('None')

n_after = len(df_clean)
print(f"Rows before duplicate drop : {n_before}")
print(f"Rows after duplicate drop  : {n_after}")
print(f"Dropped duplicate rows     : {n_before - n_after}")

Rows before duplicate drop : 48204
Rows after duplicate drop  : 48187
Dropped duplicate rows     : 17


In [14]:
# Sensor Anomaly Removal
invalid_temp = (df_clean['temp'] < 100).sum()
invalid_rain = (df_clean['rain_1h'] > 1000).sum()

print(f"Invalid temp records (< 100 K)   : {invalid_temp}")
print(f"Invalid rain records (> 1000 mm) : {invalid_rain}")

df_clean = df_clean[(df_clean['temp'] >= 100) & (df_clean['rain_1h'] <= 1000)].copy()
df_clean.reset_index(drop=True, inplace=True)

print(f"Cleaned dataset shape: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns")

Invalid temp records (< 100 K)   : 10
Invalid rain records (> 1000 mm) : 1
Cleaned dataset shape: 48176 rows x 9 columns


## Section 6 — Feature Engineering

Traffic volume exhibits strong cyclical behavior across hours of the day, days of the week, and seasons. We extract meaningful temporal features from `date_time`:

| Feature | Description / Formula | Justification |
|---------|-----------------------|---------------|
| `temp_celsius` | `temp - 273.15` | Converts temperature from Kelvin to Celsius for readability |
| `hour` | `date_time.dt.hour` (0–23) | Captures daily peak rush hours vs. low night traffic |
| `day_of_week` | `date_time.dt.dayofweek` (0–6) | Differentiates weekday commute patterns from weekend leisure travel |
| `month` | `date_time.dt.month` (1–12) | Captures seasonal variation across months |
| `is_weekend` | `1` if day_of_week >= 5 else `0` | Binary indicator for Saturday/Sunday |
| `is_holiday` | `1` if holiday != 'None' else `0` | Binary indicator for national holidays |

In [15]:
df_feat = df_clean.copy()

# Parse datetime
df_feat['date_time'] = pd.to_datetime(df_feat['date_time'])

# Temporal features
df_feat['hour']        = df_feat['date_time'].dt.hour
df_feat['day_of_week'] = df_feat['date_time'].dt.dayofweek
df_feat['month']       = df_feat['date_time'].dt.month
df_feat['is_weekend']  = (df_feat['day_of_week'] >= 5).astype(int)
df_feat['is_holiday']  = (df_feat['holiday'] != 'None').astype(int)

# Temperature in Celsius
df_feat['temp_celsius'] = df_feat['temp'] - 273.15

print("Engineered dataset preview:")
display(df_feat[['date_time', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_holiday', 'temp_celsius', 'traffic_volume']].head(8))
print(f"\nFinal shape: {df_feat.shape}")

Engineered dataset preview:
            date_time  hour  ...  temp_celsius  traffic_volume
0 2012-10-02 09:00:00     9  ...       15.1300            5545
1 2012-10-02 10:00:00    10  ...       16.2100            4516
2 2012-10-02 11:00:00    11  ...       16.4300            4767
3 2012-10-02 12:00:00    12  ...       16.9800            5026
4 2012-10-02 13:00:00    13  ...       17.9900            4918
5 2012-10-02 14:00:00    14  ...       18.5700            5181
6 2012-10-02 15:00:00    15  ...       20.0200            5584
7 2012-10-02 16:00:00    16  ...       20.7100            6015

[8 rows x 8 columns]

Final shape: (48176, 15)


## Section 7 — Train / Test Split

* **Ratio:** 80 % train / 20 % test
* **`random_state=42`** for reproducible split
* Held-out test set is reserved exclusively for final evaluation.

In [16]:
numerical_features   = ['temp_celsius', 'rain_1h', 'snow_1h', 'clouds_all', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_holiday']
categorical_features = ['weather_main']
feature_cols         = numerical_features + categorical_features
target_col           = 'traffic_volume'

X = df_feat[feature_cols].copy()
y = df_feat[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print(f"X_train : {X_train.shape}  |  y_train : {y_train.shape}")
print(f"X_test  : {X_test.shape}   |  y_test  : {y_test.shape}")
print(f"Train mean traffic: {y_train.mean():,.2f}  |  Test mean traffic: {y_test.mean():,.2f}")

X_train : (38540, 10)  |  y_train : (38540,)
X_test  : (9636, 10)   |  y_test  : (9636,)
Train mean traffic: 3,259.92  |  Test mean traffic: 3,260.19


## Section 8 — Preprocessing Pipeline

Scikit-learn `ColumnTransformer` handles feature scaling and categorical encoding:
* **Numerical features (9 columns):** `StandardScaler` (zero mean, unit variance)
* **Categorical features (`weather_main`):** `OneHotEncoder(drop='first')` to prevent multi-collinearity

**Data Leakage Prevention:** `preprocessor.fit()` is called **only on `X_train`**.

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), categorical_features),
    ],
    remainder='drop'
)

# Fit ONLY on training data
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

cat_names_out = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features).tolist()
proc_feature_names = numerical_features + cat_names_out

print(f"Processed train shape : {X_train_proc.shape}")
print(f"Processed test shape  : {X_test_proc.shape}")
print(f"Processed features ({len(proc_feature_names)} total): {proc_feature_names}")

Processed train shape : (38540, 19)
Processed test shape  : (9636, 19)
Processed features (19 total): ['temp_celsius', 'rain_1h', 'snow_1h', 'clouds_all', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_holiday', 'weather_main_Clouds', 'weather_main_Drizzle', 'weather_main_Fog', 'weather_main_Haze', 'weather_main_Mist', 'weather_main_Rain', 'weather_main_Smoke', 'weather_main_Snow', 'weather_main_Squall', 'weather_main_Thunderstorm']


## Section 9 — The First 5 Regression Algorithms

We train and evaluate **the first 5 regression algorithms** on the identical training and test sets:

1. **Linear Regression** — Standard Ordinary Least Squares
2. **Ridge Regression** — L2 regularized linear model (`alpha=1.0`)
3. **Lasso Regression** — L1 regularized linear model (`alpha=1.0`)
4. **ElasticNet Regression** — Combined L1 & L2 regularized model (`alpha=1.0, l1_ratio=0.5`)
5. **Polynomial Regression (degree = 2)** — Degree-2 interaction feature expansion followed by Linear Regression

In [18]:
# Evaluation helper function
def evaluate_model(name: str, y_true, y_pred) -> dict:
    return {
        'Model': name,
        'R2'   : round(r2_score(y_true, y_pred), 4),
        'RMSE' : round(np.sqrt(mean_squared_error(y_true, y_pred)), 2),
        'MAE'  : round(mean_absolute_error(y_true, y_pred), 2),
    }

all_results    = []
trained_models = {}
all_preds      = {}

In [19]:
# 1. Linear Regression
m1 = LinearRegression()
m1.fit(X_train_proc, y_train)
p1 = m1.predict(X_test_proc)
trained_models['Linear Regression'] = m1
all_preds['Linear Regression']      = p1
all_results.append(evaluate_model('Linear Regression', y_test, p1))
print("1. Linear Regression done.")

1. Linear Regression done.


In [20]:
# 2. Ridge Regression
m2 = Ridge(alpha=1.0)
m2.fit(X_train_proc, y_train)
p2 = m2.predict(X_test_proc)
trained_models['Ridge Regression'] = m2
all_preds['Ridge Regression']      = p2
all_results.append(evaluate_model('Ridge Regression', y_test, p2))
print("2. Ridge Regression done.")

2. Ridge Regression done.


In [21]:
# 3. Lasso Regression
m3 = Lasso(alpha=1.0, max_iter=10000)
m3.fit(X_train_proc, y_train)
p3 = m3.predict(X_test_proc)
trained_models['Lasso Regression'] = m3
all_preds['Lasso Regression']      = p3
all_results.append(evaluate_model('Lasso Regression', y_test, p3))
print("3. Lasso Regression done.")

3. Lasso Regression done.


In [22]:
# 4. ElasticNet Regression
m4 = ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000)
m4.fit(X_train_proc, y_train)
p4 = m4.predict(X_test_proc)
trained_models['ElasticNet Regression'] = m4
all_preds['ElasticNet Regression']      = p4
all_results.append(evaluate_model('ElasticNet Regression', y_test, p4))
print("4. ElasticNet Regression done.")

4. ElasticNet Regression done.


In [23]:
# 5. Polynomial Regression (degree = 2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_proc)
X_test_poly  = poly.transform(X_test_proc)

m5 = LinearRegression()
m5.fit(X_train_poly, y_train)
p5 = m5.predict(X_test_poly)
trained_models['Poly Regression (deg=2)'] = m5
all_preds['Poly Regression (deg=2)'] = p5
all_results.append(evaluate_model('Poly Regression (deg=2)', y_test, p5))
print(f"5. Polynomial Regression (degree=2) done ({X_train_poly.shape[1]} polynomial features).")

5. Polynomial Regression (degree=2) done (209 polynomial features).


In [24]:
# Consolidated comparison dataframe
results_df = (
    pd.DataFrame(all_results)
    .rename(columns={'R2': 'R²'})
    .sort_values('R²', ascending=False)
    .reset_index(drop=True)
)
results_df.index += 1
results_df.index.name = 'Rank'

print("=== 5-Algorithm Model Comparison (Ranked by R²) ===")
display(results_df)

=== 5-Algorithm Model Comparison (Ranked by R²) ===
                        Model     R²      RMSE       MAE
Rank                                                    
1     Poly Regression (deg=2) 0.7152 1059.1900  819.8200
2            Lasso Regression 0.1810 1796.1200 1573.8100
3           Linear Regression 0.1809 1796.1600 1573.0200
4            Ridge Regression 0.1809 1796.1700 1573.0500
5       ElasticNet Regression 0.1619 1816.8700 1615.0200


## Section 10 — Hyperparameter Tuning

`GridSearchCV` with 5-fold cross-validation is used to tune the regularization parameter `alpha` for **Ridge** and **Lasso** regression models on the training set.

In [25]:
# 10.1 Ridge Regression Tuning
param_grid_ridge = {'alpha': [0.1, 1.0, 10.0, 100.0, 1000.0]}
gs_ridge = GridSearchCV(Ridge(), param_grid_ridge, cv=5, scoring='r2', n_jobs=-1)
gs_ridge.fit(X_train_proc, y_train)

best_ridge = gs_ridge.best_estimator_
p_ridge_tuned = best_ridge.predict(X_test_proc)

print(f"Best Ridge Alpha : {gs_ridge.best_params_['alpha']}")
print(f"Best Ridge CV R² : {gs_ridge.best_score_:.4f}")

Best Ridge Alpha : 100.0
Best Ridge CV R² : 0.1950


In [26]:
# 10.2 Lasso Regression Tuning
param_grid_lasso = {'alpha': [0.01, 0.1, 1.0, 10.0]}
gs_lasso = GridSearchCV(Lasso(max_iter=10000), param_grid_lasso, cv=5, scoring='r2', n_jobs=-1)
gs_lasso.fit(X_train_proc, y_train)

best_lasso = gs_lasso.best_estimator_
p_lasso_tuned = best_lasso.predict(X_test_proc)

print(f"Best Lasso Alpha : {gs_lasso.best_params_['alpha']}")
print(f"Best Lasso CV R² : {gs_lasso.best_score_:.4f}")

Best Lasso Alpha : 1.0
Best Lasso CV R² : 0.1950


In [27]:
# Baseline vs Tuned Comparison
tuning_results = [
    evaluate_model('Ridge (Baseline, a=1.0)', y_test, all_preds['Ridge Regression']),
    evaluate_model(f"Ridge (Tuned, a={gs_ridge.best_params_['alpha']})", y_test, p_ridge_tuned),
    evaluate_model('Lasso (Baseline, a=1.0)', y_test, all_preds['Lasso Regression']),
    evaluate_model(f"Lasso (Tuned, a={gs_lasso.best_params_['alpha']})", y_test, p_lasso_tuned),
]
tuning_df = pd.DataFrame(tuning_results).rename(columns={'R2': 'R²'})
display(tuning_df)

                     Model     R²      RMSE       MAE
0  Ridge (Baseline, a=1.0) 0.1809 1796.1700 1573.0500
1   Ridge (Tuned, a=100.0) 0.1810 1796.0900 1573.5400
2  Lasso (Baseline, a=1.0) 0.1810 1796.1200 1573.8100
3     Lasso (Tuned, a=1.0) 0.1810 1796.1200 1573.8100


## Section 11 — 5-Fold Cross-Validation

The top two performing models (**Polynomial Regression deg=2** and **Lasso Regression**) are evaluated using 5-fold cross-validation on the training set to verify model generalization and stability.

In [28]:
top2_names = results_df.head(2)['Model'].tolist()
print(f"Top 2 models for 5-Fold CV: {top2_names}")

cv_records = []
for name in top2_names:
    if name == 'Poly Regression (deg=2)':
        scores = cross_val_score(LinearRegression(), X_train_poly, y_train, cv=5, scoring='r2')
    else:
        scores = cross_val_score(trained_models[name], X_train_proc, y_train, cv=5, scoring='r2')
    
    cv_records.append({
        'Model'   : name,
        'Fold 1'  : round(scores[0], 4),
        'Fold 2'  : round(scores[1], 4),
        'Fold 3'  : round(scores[2], 4),
        'Fold 4'  : round(scores[3], 4),
        'Fold 5'  : round(scores[4], 4),
        'Mean R²' : round(scores.mean(), 4),
        'Std Dev' : round(scores.std(), 4),
    })

cv_df = pd.DataFrame(cv_records)
print("=== 5-Fold Cross-Validation Results ===")
display(cv_df)

Top 2 models for 5-Fold CV: ['Poly Regression (deg=2)', 'Lasso Regression']
=== 5-Fold Cross-Validation Results ===
                     Model  Fold 1  Fold 2  ...  Fold 5  Mean R²  Std Dev
0  Poly Regression (deg=2)  0.7085  0.7041  ...  0.7110   0.7053   0.0039
1         Lasso Regression  0.1961  0.1828  ...  0.2093   0.1950   0.0086

[2 rows x 8 columns]


## Section 12 — Regression Visualisations

Visual diagnostic plots:
1. **Predicted vs. Actual Traffic Volume** for the best model (**Polynomial Regression deg=2**)
2. **Residual Plot** (Fitted values vs. Residuals + Residual Distribution)
3. **Model Performance Comparison Bar Charts** across all 5 algorithms

In [29]:
best_name = results_df.iloc[0]['Model']
y_pred_best = all_preds[best_name]
residuals = y_test.values - y_pred_best

print(f"Best Model: {best_name}")
print(f"  Test R²   : {r2_score(y_test, y_pred_best):.4f}")
print(f"  Test RMSE : {np.sqrt(mean_squared_error(y_test, y_pred_best)):,.2f}")
print(f"  Test MAE  : {mean_absolute_error(y_test, y_pred_best):,.2f}")

Best Model: Poly Regression (deg=2)
  Test R²   : 0.7152
  Test RMSE : 1,059.19
  Test MAE  : 819.82


In [30]:
# 12A: Predicted vs Actual
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(y_test, y_pred_best, alpha=0.3, s=15, color='steelblue', label='Predictions')
lim_lo = min(y_test.min(), y_pred_best.min()) - 100
lim_hi = max(y_test.max(), y_pred_best.max()) + 100
ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], 'r--', lw=2, label='Perfect fit (y = x)')
ax.set_xlim(lim_lo, lim_hi)
ax.set_ylim(lim_lo, lim_hi)
ax.set_xlabel('Actual Traffic Volume (vehicles/hour)', fontsize=12)
ax.set_ylabel('Predicted Traffic Volume (vehicles/hour)', fontsize=12)
ax.set_title(f'Predicted vs. Actual Traffic Volume\n{best_name}', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.text(0.05, 0.90, f"R² = {r2_score(y_test, y_pred_best):.4f}\nRMSE = {np.sqrt(mean_squared_error(y_test, y_pred_best)):.2f}",
        transform=ax.transAxes, fontsize=11, bbox=dict(facecolor='lightyellow', edgecolor='grey', boxstyle='round,pad=0.4'))

plt.tight_layout()
plt.show()

In [31]:
# 12B: Residual Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f'Residual Analysis — {best_name}', fontsize=13, fontweight='bold')

# Residuals vs Fitted
axes[0].scatter(y_pred_best, residuals, alpha=0.3, s=15, color='darkorange')
axes[0].axhline(0, color='crimson', linestyle='--', lw=2)
axes[0].set(xlabel='Fitted Values (vehicles/hour)', ylabel='Residuals', title='Residuals vs. Fitted Values')

# Residual Distribution
sns.histplot(residuals, bins=40, kde=True, ax=axes[1], color='darkorange')
axes[1].axvline(0, color='crimson', linestyle='--', lw=2)
axes[1].set(xlabel='Residual', title='Residual Distribution')

plt.tight_layout()
plt.show()

print(f"Residual mean : {residuals.mean():>10,.2f}  (near zero indicates unbiased predictions)")
print(f"Residual std  : {residuals.std():>10,.2f}")

Residual mean :      -0.40  (near zero indicates unbiased predictions)
Residual std  :   1,059.19


In [32]:
# 12C: Performance Comparison Bar Chart
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Model Performance Comparison — 5 Regression Algorithms', fontsize=14, fontweight='bold')

base5 = results_df.copy()
configs = [('R²', 'steelblue', True), ('RMSE', 'darkorange', False), ('MAE', 'mediumpurple', False)]

for ax, (metric, color, high_is_good) in zip(axes, configs):
    ordered = base5.sort_values(metric, ascending=not high_is_good)
    bars = ax.barh(ordered['Model'], ordered[metric], color=color, edgecolor='white')
    ax.set(xlabel=metric, title=f'{metric} per Model')
    if high_is_good:
        ax.invert_yaxis()
    for bar, val in zip(bars, ordered[metric]):
        label_txt = f'{val:.4f}' if metric == 'R²' else f'{val:,.1f}'
        ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height() / 2, label_txt, va='center', ha='left', fontsize=9)

plt.tight_layout()
plt.show()

## Section 13 — Final Summary and Conclusion

### Key Findings & Analytical Interpretation

1. **Best Regression Algorithm:**
   * **Polynomial Regression (degree = 2)** significantly outperforms all standard linear models, achieving an **R² of 0.7152**, **RMSE of 1,059.19**, and **MAE of 819.82** vehicles/hour on the held-out test set.

2. **Linear Baseline Models Performance:**
   * Standard **Linear Regression**, **Ridge**, and **Lasso** achieved identical test-set R² values of **0.1809–0.1810** (RMSE ~ 1,796 vehicles/hour).
   * **ElasticNet** performed slightly worse with an R² of **0.1619**.
   * This vast gap (0.181 vs 0.715 R²) demonstrates that traffic volume has strong non-linear and quadratic relationships with hour of day and weekday/weekend patterns that first-order linear models cannot capture without polynomial expansion.

3. **Hyperparameter Tuning Impact:**
   * Tuning `alpha` for Ridge (`alpha=100.0`) and Lasso (`alpha=1.0`) yielded marginal CV improvement (CV R² ~ 0.1950), confirming that first-order regularization alone cannot overcome model underfitting for non-linear traffic dynamics.

4. **Cross-Validation & Model Generalization:**
   * 5-fold cross-validation of Polynomial Regression on the training set produced a mean R² of **0.7053 ± 0.0039**, matching the test-set performance (0.7152) closely and confirming high stability without overfitting.

5. **Conclusion & Recommendation:**
   * Polynomial Regression (degree = 2) is the superior model among the 5 algorithms evaluated, capturing peak rush-hour interactions and diurnal flow dynamics cleanly.